In [ ]:
%python
# ============================================================
# CONFIGURE ME: set your catalog name before running
# ============================================================
catalog = "dbacademy"
schema  = "lr_genie_demo"

# ============================================================
# Setup - no need to edit below this line
# ============================================================

In [ ]:
%python
import os
import pandas as pd
from pyspark.sql.functions import *

base_path = os.getcwd() + "/data"
regions = ["au", "ca"]
tables  = ["customers", "opportunities", "orders"]

# Create schema (catalog already exists)
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}")
print(f"✅ Created {catalog}.{schema}")

# Load each CSV and save as table with region suffix
for region in regions:
    for table in tables:
        csv_path = f"{base_path}/genie_course_{region}/{table}.csv"
        table_name = f"{catalog}.{schema}.{table}_{region}"
        
        df = spark.createDataFrame(pd.read_csv(csv_path))
        df.write.mode("overwrite").saveAsTable(table_name)
        print(f"✅ Loaded {table_name}")

print(f"\n🎉 Setup complete! Your tables are ready in {catalog}.{schema}")

In [ ]:
%python
try:
    spark.sql(f"GRANT USAGE ON SCHEMA {catalog}.{schema} TO `account users`")
except Exception as e:
    print(f"⚠️ Could not grant schema usage (may need admin): {e}")

for region in ["au", "ca"]:
    for table in ["customers", "opportunities", "orders"]:
        try:
            spark.sql(f"GRANT SELECT ON TABLE {catalog}.{schema}.{table}_{region} TO `account users`")
            print(f"✅ Granted SELECT on {table}_{region}")
        except Exception as e:
            print(f"⚠️ Could not grant on {table}_{region}: {e}")

In [0]:
%python
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {catalog}.{schema}.meta (
        owner  STRING,
        object STRING,
        key    STRING,
        value  STRING
    )
""")

In [0]:
%python
spark.sql(f"""
    INSERT INTO {catalog}.{schema}.meta (owner, object, key, value)
    SELECT 'account users', NULL, 
        'datasets.databricks_simulated_canada_sales_and_opportunities_data', 
        '{catalog}.{schema}'
    WHERE NOT EXISTS (
        SELECT 1 FROM {catalog}.{schema}.meta
        WHERE key = 'datasets.databricks_simulated_canada_sales_and_opportunities_data'
    )
""")

spark.sql(f"""
    INSERT INTO {catalog}.{schema}.meta (owner, object, key, value)
    SELECT 'account users', NULL, 
        'datasets.databricks_simulated_australia_sales_and_opportunities_data', 
        '{catalog}.{schema}'
    WHERE NOT EXISTS (
        SELECT 1 FROM {catalog}.{schema}.meta
        WHERE key = 'datasets.databricks_simulated_australia_sales_and_opportunities_data'
    )
""")

In [0]:
%python
for region, label in [("ca", "canada"), ("au", "australia")]:
    key = f"datasets.databricks_simulated_{label}_sales_and_opportunities_data"
    value = f"{catalog}_databricks_simulated_{label}_sales_and_opportunities_data.v01"
    
    spark.sql(f"""
        INSERT INTO {catalog}.{schema}.meta (owner, object, key, value)
        SELECT 'account users', NULL, '{key}', '{value}'
        WHERE NOT EXISTS (
            SELECT 1 FROM {catalog}.{schema}.meta
            WHERE key = '{key}'
        )
    """)
    print(f"✅ Inserted meta entry for {label}")

In [0]:
%python
# Get current user and derived identifiers
current_user = spark.sql("SELECT current_user()").collect()[0][0]
pseudonym = current_user.split("@")[0].replace(".", "_")
working_dir_user = current_user.replace(".", "_")

# Check whether any meta rows already exist for this owner
owner_exists = spark.sql(
    f"SELECT 1 FROM {catalog}.{schema}.meta WHERE owner = '{current_user}' LIMIT 1"
).head(1)

# Insert meta rows only if the owner is not present
if not owner_exists:
    for key, value in [
        ('username',          current_user),
        ('catalog_name',      catalog),
        ('schema_name',       pseudonym),
        ('pseudonym',         pseudonym),
        ('paths.working_dir', f'/Volumes/{catalog}/{schema}/{working_dir_user}'),
        ('warehouse_name',    'shared_warehouse'),
    ]:
        spark.sql(
            f"INSERT INTO {catalog}.{schema}.meta (owner, object, key, value) "
            f"VALUES ('{current_user}', NULL, '{key}', '{value}')"
        )

    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{pseudonym}")
    spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog}.{schema}.{pseudonym}")
    print(f"✅ Registered {current_user}")
else:
    print(f"ℹ️ {current_user} already registered, skipping")